In [1]:
# !git clone https://github.com/correlllab/magpie_control.git
# !pip install -e magpie_control
from magpie_control import realsense_wrapper as real
import numpy as np
from PIL import Image

devices = real.poll_devices()

rsc = real.RealSense(fps=15, w=640, h=480, device_name="D405")
rsc.initConnection(device_serial=devices['D405'])

workspace_rs = real.RealSense(zMax=5, fps=6, w=640, h=480, device_name="D435")
workspace_rs.initConnection(device_serial=devices['D435'])


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# from magpie_perception.label_dino import LabelDINO
from magpie_perception.label_owlv2 import LabelOWLv2
label_vit = LabelOWLv2(topk=3)
label_vit.init()

/home/will/miniconda3/envs/octo/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
2025-03-13 14:50:35.908135: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-13 14:50:35.908199: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-13 14:50:36.007604: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-13 14:50:36.194752: I te

In [3]:
import sys
# sys.path.append("../")
# from src.magpie_perception.mask_sam2 import MaskSAM2
from magpie_perception.mask_sam2 import MaskSAM2
mask_sam2 = MaskSAM2("facebook/sam2.1-hiera-large")

In [102]:
p, rgbd_image = rsc.getPCD()
image = np.array(rgbd_image.color)
image = Image.fromarray(image)
# this is taking the image


In [103]:
# this is running it through owl v2
import numpy as np
from PIL import Image
# image = Image.open("test.jpg").convert("RGB")
image = np.array(image)
label_vit.set_threshold(0.001)
queries = ["a red block", "a blue block", "yellow block"]
queries = ["bag handle"]
abbrevq = ["bag handle"]
results, boxes, scores, labels = label_vit.label(image, queries, abbrevq, plot=True, topk=True)



In [104]:
# plots the image and bounding boxes of whichever index we assigned 
# make sure to close the image 
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use('TkAgg')
label_vit.plot_predictions(index=0) #this is index 0 because we want to get the bag handle with the highest confidence
plt.show()

In [105]:
# gets the masks of the 2 highest confidence labels
mask_sam2.set_image_and_labels(np.array(rgbd_image.color), label_vit.sorted_boxes_coords[:4 ], label_vit.sorted_labels)
masks = mask_sam2.get_masks(labels)

In [106]:
# plots the 3 masks of the three highest confidence labels.
matplotlib.use('TkAgg')
mask_sam2.plot_image(np.array(rgbd_image.color), masks, label_vit.sorted_boxes_coords, label_vit.sorted_scores[:2])
plt.show()

In [107]:
# dump rgbd image, masks, boxes, scores, labels to disk
import time
t = time.time()
object_name = queries[0]
filepath = f"bag_handle_data/{object_name}_{t}"

# make filepath if it doesnt exist
import os
os.makedirs(filepath, exist_ok=True)

# save rgbd image
np.save(filepath + "/rgbd_color.npy", np.array(rgbd_image.color))
np.save(filepath + "/rgbd_depth.npy", np.array(rgbd_image.depth))
np.save(filepath + "/masks.npy", masks)
np.save(filepath + "/boxes.npy", boxes)
np.save(filepath + "/scores.npy", scores)
np.save(filepath + "/labels.npy", labels)
np.save(filepath + "/intrinsic_matrix.npy", rsc.pinholeInstrinsics.intrinsic_matrix)
np.save(filepath + "/extrinsic_matrix.npy", rsc.extrinsics)

In [108]:
import open3d as o3d

color = np.load(filepath + "/rgbd_color.npy", allow_pickle=True)
depth = np.load(filepath + "/rgbd_depth.npy", allow_pickle=True)

depth_o3d = o3d.geometry.Image((color).astype(np.uint8))
rgb_o3d = o3d.geometry.Image((color).astype(np.uint8))
new_rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(rgb_o3d, depth_o3d)

from magpie_perception import pcd
index = 0
rgbd_image, mcpcd, tmat, pca = pcd.get_segment(mask_sam2.masks.astype(bool), 
                                          index, 
                                          rgbd_image, 
                                        #   new_rgbd, 
                                          rsc, 
                                          type="mask", 
                                          viz_scale=2500.0, 
                                          display=True,
                                          method="iterative")

z-axis dot product: [0.90556964]
[Open3D INFO] Window window_8 created.


WebVisualizer(window_uid='window_8')